# Marketplace Growth & Customer Experience Analysis

## C3. Cleaned Data Validation

### Objective

This notebook validates the cleaned Olist datasets produced by `C2_data_cleaning.ipynb` before database design and loading.

The validation covers:

- Row and column reconciliation
- Final schema and data types
- Primary and composite keys
- Exact duplicate records
- Referential integrity
- Table grain and relationship cardinality
- Missing-value behaviour
- Structural and formatting transformations
- Categorical and numerical domains
- Retained temporal anomalies
- Lookup and geographic coverage
- Database-readiness status

This notebook does not modify the cleaned data. Failed critical checks must be resolved before proceeding to database design.

## Validation Boundaries

This notebook validates the cleaned datasets without repeating the complete raw-data audit or applying additional transformations.

Potential business anomalies that were deliberately retained during cleaning are documented as limitations rather than automatically treated as validation failures.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

C2_NOTEBOOK = Path("C2_data_cleaning.ipynb")

if not C2_NOTEBOOK.exists():
    raise FileNotFoundError(
        f"Cleaning notebook not found: {C2_NOTEBOOK.resolve()}"
    )

## 2. Load the Cleaned Data

The cleaning notebook is executed so that C3 validates the exact DataFrames and transformation logic produced by C2.

Its displayed outputs are captured to keep this notebook focused on validation.

In [2]:
%%capture c2_execution_output

%run ./C2_data_cleaning.ipynb

required_objects = [
    "datasets_raw",
    "cleaned_datasets",
    "baseline_rows",
    "baseline_columns",
    "datetime_columns",
    "zip_columns",
    "text_columns",
    "product_integer_columns",
    "geolocation_rows_removed",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "C2 did not create the following required objects: "
        + ", ".join(missing_objects)
    )

required_tables = {
    "customers",
    "orders",
    "order_items",
    "payments",
    "reviews",
    "products",
    "sellers",
    "geolocation",
    "category_translation",
}

available_tables = set(cleaned_datasets)

missing_tables = required_tables - available_tables
unexpected_tables = available_tables - required_tables

if missing_tables:
    raise KeyError(
        "Missing cleaned datasets: "
        + ", ".join(sorted(missing_tables))
    )

if unexpected_tables:
    raise KeyError(
        "Unexpected cleaned datasets: "
        + ", ".join(sorted(unexpected_tables))
    )

print("All required cleaned datasets are available.")

In [3]:
cleaned_inventory = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "missing_cells": int(
                dataframe.isna().sum().sum()
            ),
            "exact_duplicate_rows": int(
                dataframe.duplicated().sum()
            ),
        }
        for table_name, dataframe
        in cleaned_datasets.items()
    ]
)

display(cleaned_inventory)

,table_name,rows,columns,missing_cells,exact_duplicate_rows
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145932,0
5,products,32951,9,2448,0
6,sellers,3095,4,0,0
7,geolocation,738332,5,0,0
8,category_translation,71,2,0,0


## 3. Initialise the Validation Register

Each test is recorded in a common register.

Validation severity is classified as:

- **Critical** — failure prevents database readiness.
- **Warning** — limitation must be documented but does not necessarily prevent loading.

In [4]:
validation_results = []


def record_validation(
    area,
    table_name,
    check,
    result,
    details,
    severity="Critical",
):
    """Add one validation result to the register."""

    valid_results = {
        "Passed",
        "Passed with limitation",
        "Failed",
    }

    valid_severities = {
        "Critical",
        "Warning",
    }

    if result not in valid_results:
        raise ValueError(f"Invalid validation result: {result}")

    if severity not in valid_severities:
        raise ValueError(f"Invalid severity: {severity}")

    validation_results.append(
        {
            "validation_area": area,
            "table_name": table_name,
            "check": check,
            "severity": severity,
            "result": result,
            "details": details,
        }
    )

## 4. Row and Column Reconciliation

All cleaned tables should retain their original number of rows and columns except geolocation, where confirmed exact duplicate rows were removed.

Renaming columns changes their labels but not the total number of columns.

In [5]:
reconciliation_rows = []

for table_name, dataframe in cleaned_datasets.items():
    expected_rows = baseline_rows[table_name]

    if table_name == "geolocation":
        expected_rows -= geolocation_rows_removed

    expected_columns = baseline_columns[table_name]

    actual_rows = len(dataframe)
    actual_columns = dataframe.shape[1]

    row_check_passed = actual_rows == expected_rows
    column_check_passed = actual_columns == expected_columns

    reconciliation_rows.append(
        {
            "table_name": table_name,
            "raw_rows": baseline_rows[table_name],
            "expected_cleaned_rows": expected_rows,
            "actual_cleaned_rows": actual_rows,
            "row_difference_from_expected": (
                actual_rows - expected_rows
            ),
            "expected_columns": expected_columns,
            "actual_columns": actual_columns,
        }
    )

    record_validation(
        area="Reconciliation",
        table_name=table_name,
        check="Expected row count",
        result="Passed" if row_check_passed else "Failed",
        details=(
            f"Expected {expected_rows:,}; "
            f"found {actual_rows:,}"
        ),
    )

    record_validation(
        area="Reconciliation",
        table_name=table_name,
        check="Expected column count",
        result=(
            "Passed"
            if column_check_passed
            else "Failed"
        ),
        details=(
            f"Expected {expected_columns}; "
            f"found {actual_columns}"
        ),
    )

reconciliation_summary = pd.DataFrame(
    reconciliation_rows
)

display(reconciliation_summary)

,table_name,raw_rows,expected_cleaned_rows,actual_cleaned_rows,row_difference_from_expected,expected_columns,actual_columns
0,customers,99441,99441,99441,0,5,5
1,orders,99441,99441,99441,0,8,8
2,order_items,112650,112650,112650,0,7,7
3,payments,103886,103886,103886,0,5,5
4,reviews,99224,99224,99224,0,7,7
5,products,32951,32951,32951,0,9,9
6,sellers,3095,3095,3095,0,4,4
7,geolocation,1000163,738332,738332,0,5,5
8,category_translation,71,71,71,0,2,2


## 5. Final Schema Validation

Validate that every cleaned table contains the expected columns in the expected order.

The expected schema represents the cleaned layer, not the original raw CSV structure.

In [6]:
expected_schema = {
    "customers": {
        "customer_id": "string",
        "customer_unique_id": "string",
        "customer_zip_code_prefix": "string",
        "customer_city": "string",
        "customer_state": "string",
    },
    "orders": {
        "order_id": "string",
        "customer_id": "string",
        "order_status": "string",
        "order_purchase_timestamp": "datetime",
        "order_approved_at": "datetime",
        "order_delivered_carrier_date": "datetime",
        "order_delivered_customer_date": "datetime",
        "order_estimated_delivery_date": "datetime",
    },
    "order_items": {
        "order_id": "string",
        "order_item_id": "integer",
        "product_id": "string",
        "seller_id": "string",
        "shipping_limit_date": "datetime",
        "price": "numeric",
        "freight_value": "numeric",
    },
    "payments": {
        "order_id": "string",
        "payment_sequential": "integer",
        "payment_type": "string",
        "payment_installments": "integer",
        "payment_value": "numeric",
    },
    "reviews": {
        "review_id": "string",
        "order_id": "string",
        "review_score": "integer",
        "review_comment_title": "string",
        "review_comment_message": "string",
        "review_creation_date": "datetime",
        "review_answer_timestamp": "datetime",
    },
    "products": {
        "product_id": "string",
        "product_category_name": "string",
        "product_name_length": "nullable_integer",
        "product_description_length": "nullable_integer",
        "product_photos_qty": "nullable_integer",
        "product_weight_g": "numeric",
        "product_length_cm": "numeric",
        "product_height_cm": "numeric",
        "product_width_cm": "numeric",
    },
    "sellers": {
        "seller_id": "string",
        "seller_zip_code_prefix": "string",
        "seller_city": "string",
        "seller_state": "string",
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "string",
        "geolocation_lat": "numeric",
        "geolocation_lng": "numeric",
        "geolocation_city": "string",
        "geolocation_state": "string",
    },
    "category_translation": {
        "product_category_name": "string",
        "product_category_name_english": "string",
    },
}

def dtype_matches(series, expected_type):
    """Check whether a Series matches an expected type group."""

    if expected_type == "string":
        return str(series.dtype).startswith("string")

    if expected_type == "datetime":
        return pd.api.types.is_datetime64_any_dtype(
            series
        )

    if expected_type == "integer":
        return pd.api.types.is_integer_dtype(series)

    if expected_type == "nullable_integer":
        return str(series.dtype) == "Int64"

    if expected_type == "numeric":
        return pd.api.types.is_numeric_dtype(series)

    raise ValueError(
        f"Unsupported expected type: {expected_type}"
    )

schema_validation_rows = []
dtype_validation_rows = []

for table_name, expected_columns in expected_schema.items():
    dataframe = cleaned_datasets[table_name]

    expected_column_names = list(expected_columns)
    actual_column_names = dataframe.columns.tolist()

    missing_columns = [
        column
        for column in expected_column_names
        if column not in actual_column_names
    ]

    unexpected_columns = [
        column
        for column in actual_column_names
        if column not in expected_column_names
    ]

    column_order_matches = (
        actual_column_names == expected_column_names
    )

    structure_passed = (
        not missing_columns
        and not unexpected_columns
        and column_order_matches
    )

    schema_validation_rows.append(
        {
            "table_name": table_name,
            "missing_columns": ", ".join(missing_columns),
            "unexpected_columns": ", ".join(
                unexpected_columns
            ),
            "column_order_matches": column_order_matches,
            "schema_passed": structure_passed,
        }
    )

    record_validation(
        area="Schema",
        table_name=table_name,
        check="Expected columns and order",
        result=(
            "Passed"
            if structure_passed
            else "Failed"
        ),
        details=(
            f"Missing: {missing_columns or 'None'}; "
            f"unexpected: {unexpected_columns or 'None'}; "
            f"order matches: {column_order_matches}"
        ),
    )

    for column_name, expected_type in expected_columns.items():
        if column_name not in dataframe.columns:
            continue

        actual_dtype = str(dataframe[column_name].dtype)

        type_passed = dtype_matches(
            dataframe[column_name],
            expected_type,
        )

        dtype_validation_rows.append(
            {
                "table_name": table_name,
                "column_name": column_name,
                "expected_type": expected_type,
                "actual_dtype": actual_dtype,
                "type_matches": type_passed,
            }
        )

schema_validation = pd.DataFrame(
    schema_validation_rows
)

dtype_validation = pd.DataFrame(
    dtype_validation_rows
)

display(schema_validation)
display(dtype_validation)

,table_name,missing_columns,unexpected_columns,column_order_matches,schema_passed
0,customers,,,True,True
1,orders,,,True,True
2,order_items,,,True,True
3,payments,,,True,True
4,reviews,,,True,True
5,products,,,True,True
6,sellers,,,True,True
7,geolocation,,,True,True
8,category_translation,,,True,True


,table_name,column_name,expected_type,actual_dtype,type_matches
0,customers,customer_id,string,string,True
1,customers,customer_unique_id,string,string,True
2,customers,customer_zip_code_prefix,string,string,True
3,customers,customer_city,string,string,True
4,customers,customer_state,string,string,True
5,orders,order_id,string,string,True
6,orders,customer_id,string,string,True
7,orders,order_status,string,string,True
8,orders,order_purchase_timestamp,datetime,datetime64[ns],True
9,orders,order_approved_at,datetime,datetime64[ns],True


In [7]:
for table_name in expected_schema:
    table_type_results = dtype_validation.loc[
        dtype_validation["table_name"] == table_name
    ]

    type_check_passed = (
        not table_type_results.empty
        and table_type_results["type_matches"].all()
    )

    mismatched_columns = table_type_results.loc[
        ~table_type_results["type_matches"],
        "column_name",
    ].tolist()

    record_validation(
        area="Data types",
        table_name=table_name,
        check="Expected cleaned data types",
        result=(
            "Passed"
            if type_check_passed
            else "Failed"
        ),
        details=(
            "All expected types matched"
            if type_check_passed
            else "Mismatched columns: "
            + ", ".join(mismatched_columns)
        ),
    )

## 6. Key and Grain Validation

Validate the identifiers that define the grain of each cleaned table.

A proposed key must be complete and unique.

In [8]:
key_definitions = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": [
        "order_id",
        "order_item_id",
    ],
    "payments": [
        "order_id",
        "payment_sequential",
    ],
    "reviews": [
        "review_id",
        "order_id",
    ],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": [
        "product_category_name",
    ],
}

key_validation_rows = []

for table_name, key_columns in key_definitions.items():
    dataframe = cleaned_datasets[table_name]

    missing_key_rows = int(
        dataframe[key_columns]
        .isna()
        .any(axis=1)
        .sum()
    )

    duplicate_key_rows = int(
        dataframe
        .duplicated(subset=key_columns)
        .sum()
    )

    key_passed = (
        missing_key_rows == 0
        and duplicate_key_rows == 0
    )

    key_validation_rows.append(
        {
            "table_name": table_name,
            "key_columns": " + ".join(key_columns),
            "missing_key_rows": missing_key_rows,
            "duplicate_key_rows": duplicate_key_rows,
            "key_valid": key_passed,
        }
    )

    record_validation(
        area="Key integrity",
        table_name=table_name,
        check="Key completeness and uniqueness",
        result="Passed" if key_passed else "Failed",
        details=(
            f"Missing-key rows: {missing_key_rows:,}; "
            f"duplicate-key rows: {duplicate_key_rows:,}"
        ),
    )

key_validation = pd.DataFrame(
    key_validation_rows
)

display(key_validation)

,table_name,key_columns,missing_key_rows,duplicate_key_rows,key_valid
0,customers,customer_id,0,0,True
1,orders,order_id,0,0,True
2,order_items,order_id + order_item_id,0,0,True
3,payments,order_id + payment_sequential,0,0,True
4,reviews,review_id + order_id,0,0,True
5,products,product_id,0,0,True
6,sellers,seller_id,0,0,True
7,category_translation,product_category_name,0,0,True


In [9]:
identifier_repetition_summary = pd.DataFrame(
    [
        {
            "table_name": "customers",
            "identifier": "customer_unique_id",
            "rows_with_repeated_identifier": int(
                cleaned_datasets["customers"][
                    "customer_unique_id"
                ]
                .duplicated(keep=False)
                .sum()
            ),
            "interpretation": (
                "Expected business identity repetition"
            ),
        },
        {
            "table_name": "reviews",
            "identifier": "review_id",
            "rows_with_repeated_identifier": int(
                cleaned_datasets["reviews"][
                    "review_id"
                ]
                .duplicated(keep=False)
                .sum()
            ),
            "interpretation": (
                "Allowed because review_id + order_id "
                "is unique"
            ),
        },
    ]
)

display(identifier_repetition_summary)

for _, row in identifier_repetition_summary.iterrows():
    record_validation(
        area="Key interpretation",
        table_name=row["table_name"],
        check=(
            f"Repeated {row['identifier']} values"
        ),
        result="Passed with limitation",
        details=(
            f"{int(row['rows_with_repeated_identifier']):,} "
            f"rows; {row['interpretation']}"
        ),
        severity="Warning",
    )

,table_name,identifier,rows_with_repeated_identifier,interpretation
0,customers,customer_unique_id,6342,Expected business identity repetition
1,reviews,review_id,1603,Allowed because review_id + order_id is unique


## 7. Exact Duplicate Validation

Confirm that no fully identical rows remain after cleaning.

In [10]:
duplicate_validation_rows = []

for table_name, dataframe in cleaned_datasets.items():
    duplicate_rows = int(
        dataframe.duplicated().sum()
    )

    check_passed = duplicate_rows == 0

    duplicate_validation_rows.append(
        {
            "table_name": table_name,
            "exact_duplicate_rows": duplicate_rows,
            "validation_passed": check_passed,
        }
    )

    record_validation(
        area="Duplicate records",
        table_name=table_name,
        check="No exact duplicate rows",
        result="Passed" if check_passed else "Failed",
        details=(
            f"{duplicate_rows:,} exact duplicate rows"
        ),
    )

duplicate_validation = pd.DataFrame(
    duplicate_validation_rows
)

display(duplicate_validation)

,table_name,exact_duplicate_rows,validation_passed
0,customers,0,True
1,orders,0,True
2,order_items,0,True
3,payments,0,True
4,reviews,0,True
5,products,0,True
6,sellers,0,True
7,geolocation,0,True
8,category_translation,0,True


## 8. Referential Integrity

Validate core parent-child relationships independently.

Each relationship is tested at its documented grain to avoid creating analytical fanout.

In [11]:
relationship_definitions = [
    {
        "relationship": "orders → customers",
        "child_table": "orders",
        "child_key": "customer_id",
        "parent_table": "customers",
        "parent_key": "customer_id",
    },
    {
        "relationship": "order_items → orders",
        "child_table": "order_items",
        "child_key": "order_id",
        "parent_table": "orders",
        "parent_key": "order_id",
    },
    {
        "relationship": "order_items → products",
        "child_table": "order_items",
        "child_key": "product_id",
        "parent_table": "products",
        "parent_key": "product_id",
    },
    {
        "relationship": "order_items → sellers",
        "child_table": "order_items",
        "child_key": "seller_id",
        "parent_table": "sellers",
        "parent_key": "seller_id",
    },
    {
        "relationship": "payments → orders",
        "child_table": "payments",
        "child_key": "order_id",
        "parent_table": "orders",
        "parent_key": "order_id",
    },
    {
        "relationship": "reviews → orders",
        "child_table": "reviews",
        "child_key": "order_id",
        "parent_table": "orders",
        "parent_key": "order_id",
    },
]

referential_integrity_rows = []

for relationship in relationship_definitions:
    child_values = (
        cleaned_datasets[
            relationship["child_table"]
        ][relationship["child_key"]]
        .dropna()
    )

    parent_values = (
        cleaned_datasets[
            relationship["parent_table"]
        ][relationship["parent_key"]]
        .dropna()
    )

    orphan_rows = int(
        (~child_values.isin(parent_values)).sum()
    )

    check_passed = orphan_rows == 0

    referential_integrity_rows.append(
        {
            **relationship,
            "orphan_rows": orphan_rows,
            "validation_passed": check_passed,
        }
    )

    record_validation(
        area="Referential integrity",
        table_name=relationship["child_table"],
        check=relationship["relationship"],
        result="Passed" if check_passed else "Failed",
        details=f"{orphan_rows:,} orphan rows",
    )

referential_integrity = pd.DataFrame(
    referential_integrity_rows
)

display(referential_integrity)

,relationship,child_table,child_key,parent_table,parent_key,orphan_rows,validation_passed
0,orders → customers,orders,customer_id,customers,customer_id,0,True
1,order_items → orders,order_items,order_id,orders,order_id,0,True
2,order_items → products,order_items,product_id,products,product_id,0,True
3,order_items → sellers,order_items,seller_id,sellers,seller_id,0,True
4,payments → orders,payments,order_id,orders,order_id,0,True
5,reviews → orders,reviews,order_id,orders,order_id,0,True


## 9. Parent Coverage and Relationship Cardinality

Parent records without corresponding child records are measured separately from orphan records.

Their presence may represent valid business states and does not automatically indicate a failed relationship.

In [12]:
coverage_definitions = [
    {
        "relationship": "customers without orders",
        "parent_table": "customers",
        "parent_key": "customer_id",
        "child_table": "orders",
        "child_key": "customer_id",
    },
    {
        "relationship": "orders without items",
        "parent_table": "orders",
        "parent_key": "order_id",
        "child_table": "order_items",
        "child_key": "order_id",
    },
    {
        "relationship": "orders without payments",
        "parent_table": "orders",
        "parent_key": "order_id",
        "child_table": "payments",
        "child_key": "order_id",
    },
    {
        "relationship": "orders without reviews",
        "parent_table": "orders",
        "parent_key": "order_id",
        "child_table": "reviews",
        "child_key": "order_id",
    },
    {
        "relationship": "products without order items",
        "parent_table": "products",
        "parent_key": "product_id",
        "child_table": "order_items",
        "child_key": "product_id",
    },
    {
        "relationship": "sellers without order items",
        "parent_table": "sellers",
        "parent_key": "seller_id",
        "child_table": "order_items",
        "child_key": "seller_id",
    },
]

coverage_rows = []

for definition in coverage_definitions:
    parent_values = cleaned_datasets[
        definition["parent_table"]
    ][definition["parent_key"]]

    child_values = cleaned_datasets[
        definition["child_table"]
    ][definition["child_key"]]

    uncovered_rows = int(
        (~parent_values.isin(child_values)).sum()
    )

    coverage_rows.append(
        {
            "relationship": definition["relationship"],
            "parent_rows_without_child": uncovered_rows,
        }
    )

    record_validation(
        area="Relationship coverage",
        table_name=definition["parent_table"],
        check=definition["relationship"],
        result=(
            "Passed"
            if uncovered_rows == 0
            else "Passed with limitation"
        ),
        details=(
            f"{uncovered_rows:,} parent rows "
            "without a child record"
        ),
        severity="Warning",
    )

parent_coverage = pd.DataFrame(coverage_rows)

display(parent_coverage)

,relationship,parent_rows_without_child
0,customers without orders,0
1,orders without items,775
2,orders without payments,1
3,orders without reviews,768
4,products without order items,0
5,sellers without order items,0


In [13]:
def summarise_group_size(series, relationship):
    """Summarise a grouped relationship count."""

    return {
        "relationship": relationship,
        "minimum": int(series.min()),
        "median": float(series.median()),
        "maximum": int(series.max()),
        "average": round(float(series.mean()), 2),
    }


cardinality_summary = pd.DataFrame(
    [
        summarise_group_size(
            cleaned_datasets["orders"]
            .groupby("customer_id")
            .size(),
            "orders per customer record",
        ),
        summarise_group_size(
            cleaned_datasets["customers"]
            .groupby("customer_unique_id")
            .size(),
            "customer records per unique customer",
        ),
        summarise_group_size(
            cleaned_datasets["order_items"]
            .groupby("order_id")
            .size(),
            "items per order",
        ),
        summarise_group_size(
            cleaned_datasets["payments"]
            .groupby("order_id")
            .size(),
            "payments per order",
        ),
        summarise_group_size(
            cleaned_datasets["reviews"]
            .groupby("order_id")
            .size(),
            "reviews per order",
        ),
        summarise_group_size(
            cleaned_datasets["order_items"]
            .groupby("order_id")["seller_id"]
            .nunique(),
            "unique sellers per order",
        ),
    ]
)

display(cardinality_summary)

,relationship,minimum,median,maximum,average
0,orders per customer record,1,1.00,1,1.00
1,customer records per unique customer,1,1.00,17,1.03
2,items per order,1,1.00,21,1.14
3,payments per order,1,1.00,29,1.04
4,reviews per order,1,1.00,3,1.01
5,unique sellers per order,1,1.00,5,1.01


### Relationship Validation Summary

The cleaned datasets preserve the relationship structure identified during the raw-data audit.

Key observations are:

- Every order references a valid customer record.
- All order items reference valid orders, products, and sellers.
- All payment and review records reference valid orders.
- `customer_id` remains one-to-one with an order record in this dataset, while `customer_unique_id` can represent repeat customers across multiple records.
- Orders may contain multiple items, payments, reviews, and sellers.

Parent records without children are retained where they represent valid or unresolved business states. In particular, some orders have no items, payments, or reviews.

These relationship patterns must be respected during later SQL joins to prevent row multiplication and metric inflation.

## 10. Missing-Value Validation

Missing values are permitted only in columns already classified as optional, conditionally applicable, or unavailable from the source.

Any missing value outside the approved columns is treated as a critical validation failure.

In [14]:
allowed_missing_columns = {
    "customers": set(),
    "orders": {
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    },
    "order_items": set(),
    "payments": set(),
    "reviews": {
        "review_comment_title",
        "review_comment_message",
    },
    "products": {
        "product_category_name",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    },
    "sellers": set(),
    "geolocation": set(),
    "category_translation": set(),
}

missing_value_rows = []

for table_name, dataframe in cleaned_datasets.items():
    missing_counts = dataframe.isna().sum()

    columns_with_missing = set(
        missing_counts[
            missing_counts > 0
        ].index
    )

    unexpected_missing_columns = sorted(
        columns_with_missing
        - allowed_missing_columns[table_name]
    )

    missing_check_passed = (
        not unexpected_missing_columns
    )

    for column_name in sorted(columns_with_missing):
        missing_value_rows.append(
            {
                "table_name": table_name,
                "column_name": column_name,
                "missing_count": int(
                    missing_counts[column_name]
                ),
                "missing_pct": round(
                    missing_counts[column_name]
                    / len(dataframe)
                    * 100,
                    2,
                ),
                "missing_allowed": (
                    column_name
                    in allowed_missing_columns[table_name]
                ),
            }
        )

    record_validation(
        area="Missing values",
        table_name=table_name,
        check="Missing values limited to approved columns",
        result=(
            "Passed"
            if missing_check_passed
            else "Failed"
        ),
        details=(
            "No unexpected missing values"
            if missing_check_passed
            else "Unexpected missing columns: "
            + ", ".join(unexpected_missing_columns)
        ),
    )

remaining_missing_values = pd.DataFrame(
    missing_value_rows
)

display(remaining_missing_values)

,table_name,column_name,missing_count,missing_pct,missing_allowed
0,orders,order_approved_at,160,0.16,True
1,orders,order_delivered_carrier_date,1783,1.79,True
2,orders,order_delivered_customer_date,2965,2.98,True
3,reviews,review_comment_message,58274,58.73,True
4,reviews,review_comment_title,87658,88.34,True
5,products,product_category_name,610,1.85,True
6,products,product_description_length,610,1.85,True
7,products,product_height_cm,2,0.01,True
8,products,product_length_cm,2,0.01,True
9,products,product_name_length,610,1.85,True


## 11. Transformation Validation

Confirm that the cleaning actions applied in C2 produced the intended structural and formatting results.

In [15]:
transformation_checks = []


def add_transformation_check(
    table_name,
    check,
    passed,
    details,
):
    transformation_checks.append(
        {
            "table_name": table_name,
            "check": check,
            "validation_passed": bool(passed),
            "details": details,
        }
    )

    record_validation(
        area="Transformation",
        table_name=table_name,
        check=check,
        result="Passed" if passed else "Failed",
        details=details,
    )


raw_product_columns_preserved = (
    "product_name_lenght"
    in datasets_raw["products"].columns
    and "product_description_lenght"
    in datasets_raw["products"].columns
)

clean_product_columns_corrected = (
    "product_name_length"
    in cleaned_datasets["products"].columns
    and "product_description_length"
    in cleaned_datasets["products"].columns
    and "product_name_lenght"
    not in cleaned_datasets["products"].columns
    and "product_description_lenght"
    not in cleaned_datasets["products"].columns
)

add_transformation_check(
    table_name="products",
    check="Raw source column names preserved",
    passed=raw_product_columns_preserved,
    details=(
        "Raw products DataFrame retains the original "
        "source spelling"
    ),
)

add_transformation_check(
    table_name="products",
    check="Cleaned product column names corrected",
    passed=clean_product_columns_corrected,
    details=(
        "Cleaned products DataFrame uses corrected "
        "column names"
    ),
)


for table_name, columns in datetime_columns.items():
    for column in columns:
        is_datetime = (
            pd.api.types.is_datetime64_any_dtype(
                cleaned_datasets[table_name][column]
            )
        )

        missing_count_preserved = (
            datasets_raw[table_name][column]
            .isna()
            .sum()
            == cleaned_datasets[table_name][column]
            .isna()
            .sum()
        )

        add_transformation_check(
            table_name=table_name,
            check=f"{column} converted to datetime",
            passed=(
                is_datetime
                and missing_count_preserved
            ),
            details=(
                f"Datetime dtype: {is_datetime}; "
                "missing count preserved: "
                f"{missing_count_preserved}"
            ),
        )


for table_name, column in zip_columns.items():
    non_null_values = (
        cleaned_datasets[table_name][column]
        .dropna()
    )

    zip_format_valid = (
        str(
            cleaned_datasets[table_name][column].dtype
        ).startswith("string")
        and non_null_values
        .str.fullmatch(r"\d{5}")
        .all()
    )

    add_transformation_check(
        table_name=table_name,
        check=f"{column} uses five-digit string format",
        passed=zip_format_valid,
        details=(
            "All non-null ZIP prefixes contain "
            "exactly five digits"
        ),
    )


raw_product_column_map = {
    "product_name_length": "product_name_lenght",
    "product_description_length": (
        "product_description_lenght"
    ),
}

for column in product_integer_columns:
    raw_column = raw_product_column_map.get(
        column,
        column,
    )

    nullable_integer_valid = (
        str(
            cleaned_datasets["products"][column].dtype
        )
        == "Int64"
    )

    missing_count_preserved = (
        datasets_raw["products"][raw_column]
        .isna()
        .sum()
        == cleaned_datasets["products"][column]
        .isna()
        .sum()
    )

    add_transformation_check(
        table_name="products",
        check=f"{column} uses nullable integer dtype",
        passed=(
            nullable_integer_valid
            and missing_count_preserved
        ),
        details=(
            f"Int64 dtype: {nullable_integer_valid}; "
            "missing count preserved: "
            f"{missing_count_preserved}"
        ),
    )


for table_name, columns in text_columns.items():
    for column in columns:
        values = (
            cleaned_datasets[table_name][column]
            .dropna()
        )

        no_surrounding_whitespace = values.eq(
            values.str.strip()
        ).all()

        no_empty_strings = not values.eq("").any()

        add_transformation_check(
            table_name=table_name,
            check=f"{column} text formatting",
            passed=(
                no_surrounding_whitespace
                and no_empty_strings
            ),
            details=(
                "No surrounding whitespace or "
                "empty strings remain"
            ),
        )


for table_name, dataframe in cleaned_datasets.items():
    remaining_object_columns = (
        dataframe.select_dtypes(
            include="object"
        ).columns.tolist()
    )

    no_object_columns = not remaining_object_columns

    add_transformation_check(
        table_name=table_name,
        check="No generic object columns remain",
        passed=no_object_columns,
        details=(
            "None"
            if no_object_columns
            else ", ".join(remaining_object_columns)
        ),
    )


transformation_validation = pd.DataFrame(
    transformation_checks
)

display(transformation_validation)

,table_name,check,validation_passed,details
0,products,Raw source column names preserved,True,Raw products DataFrame retains the original so...
1,products,Cleaned product column names corrected,True,Cleaned products DataFrame uses corrected colu...
2,orders,order_purchase_timestamp converted to datetime,True,Datetime dtype: True; missing count preserved:...
3,orders,order_approved_at converted to datetime,True,Datetime dtype: True; missing count preserved:...
4,orders,order_delivered_carrier_date converted to date...,True,Datetime dtype: True; missing count preserved:...
5,orders,order_delivered_customer_date converted to dat...,True,Datetime dtype: True; missing count preserved:...
6,orders,order_estimated_delivery_date converted to dat...,True,Datetime dtype: True; missing count preserved:...
7,order_items,shipping_limit_date converted to datetime,True,Datetime dtype: True; missing count preserved:...
8,reviews,review_creation_date converted to datetime,True,Datetime dtype: True; missing count preserved:...
9,reviews,review_answer_timestamp converted to datetime,True,Datetime dtype: True; missing count preserved:...


## 12. Categorical Domain Validation

Validate the known domains of order status, payment type, review score, and state-code fields.

In [16]:
categorical_domain_checks = [
    {
        "table_name": "orders",
        "column_name": "order_status",
        "allowed_values": {
            "approved",
            "canceled",
            "created",
            "delivered",
            "invoiced",
            "processing",
            "shipped",
            "unavailable",
        },
    },
    {
        "table_name": "payments",
        "column_name": "payment_type",
        "allowed_values": {
            "boleto",
            "credit_card",
            "debit_card",
            "not_defined",
            "voucher",
        },
    },
]

categorical_validation_rows = []

for check in categorical_domain_checks:
    actual_values = set(
        cleaned_datasets[
            check["table_name"]
        ][check["column_name"]]
        .dropna()
        .unique()
    )

    unexpected_values = sorted(
        actual_values - check["allowed_values"]
    )

    check_passed = not unexpected_values

    categorical_validation_rows.append(
        {
            "table_name": check["table_name"],
            "column_name": check["column_name"],
            "actual_values": ", ".join(
                sorted(actual_values)
            ),
            "unexpected_values": ", ".join(
                unexpected_values
            ),
            "validation_passed": check_passed,
        }
    )

    record_validation(
        area="Categorical domain",
        table_name=check["table_name"],
        check=f"{check['column_name']} allowed values",
        result="Passed" if check_passed else "Failed",
        details=(
            "No unexpected values"
            if check_passed
            else "Unexpected values: "
            + ", ".join(unexpected_values)
        ),
    )

categorical_validation = pd.DataFrame(
    categorical_validation_rows
)

display(categorical_validation)

,table_name,column_name,actual_values,unexpected_values,validation_passed
0,orders,order_status,"approved, canceled, created, delivered, invoic...",,True
1,payments,payment_type,"boleto, credit_card, debit_card, not_defined, ...",,True


In [17]:
review_score_invalid = int(
    (
        cleaned_datasets["reviews"][
            "review_score"
        ].isna()
        | ~cleaned_datasets["reviews"][
            "review_score"
        ].between(1, 5)
    ).sum()
)

record_validation(
    area="Categorical domain",
    table_name="reviews",
    check="Review scores between 1 and 5",
    result=(
        "Passed"
        if review_score_invalid == 0
        else "Failed"
    ),
    details=(
        f"{review_score_invalid:,} invalid rows"
    ),
)


valid_brazil_state_codes = {
    "AC","AL","AP","AM","BA","CE","DF","ES","GO",
    "MA","MT","MS","MG","PA","PB","PR","PE","PI",
    "RJ","RN","RS","RO","RR","SC","SP","SE","TO",
}

state_columns = {
    "customers": "customer_state",
    "sellers": "seller_state",
    "geolocation": "geolocation_state",
}

state_validation_rows = []

for table_name, column_name in state_columns.items():
    values = cleaned_datasets[
        table_name
    ][column_name]

    invalid_mask = (
        values.isna()
        | ~values.isin(valid_brazil_state_codes)
    )

    invalid_rows = int(
        invalid_mask.sum()
    )

    invalid_values = sorted(
        values.loc[
            invalid_mask & values.notna()
        ]
        .unique()
        .tolist()
    )

    check_passed = invalid_rows == 0

    state_validation_rows.append(
        {
            "table_name": table_name,
            "column_name": column_name,
            "invalid_state_rows": invalid_rows,
            "invalid_values": ", ".join(
                invalid_values
            ),
            "validation_passed": check_passed,
        }
    )

    record_validation(
        area="Categorical domain",
        table_name=table_name,
        check=(
            f"{column_name} uses valid Brazilian "
            "state codes"
        ),
        result=(
            "Passed"
            if check_passed
            else "Failed"
        ),
        details=(
            "No invalid state codes"
            if check_passed
            else (
                f"{invalid_rows:,} invalid rows; "
                "values: "
                + ", ".join(invalid_values)
            )
        ),
    )


state_validation = pd.DataFrame(
    state_validation_rows
)

display(state_validation)

,table_name,column_name,invalid_state_rows,invalid_values,validation_passed
0,customers,customer_state,0,,True
1,sellers,seller_state,0,,True
2,geolocation,geolocation_state,0,,True


## 13. Numerical Domain Validation

Validate minimum business and mathematical constraints without removing statistical outliers.

In [18]:
products_clean = cleaned_datasets["products"]
order_items_clean = cleaned_datasets["order_items"]
payments_clean = cleaned_datasets["payments"]
geolocation_clean = cleaned_datasets["geolocation"]

product_count_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
]

product_physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

numerical_checks = [
    {
        "table_name": "order_items",
        "check": "order_item_id must be at least 1",
        "invalid_rows": int(
            order_items_clean["order_item_id"]
            .lt(1)
            .sum()
        ),
    },
    {
        "table_name": "order_items",
        "check": "price must be greater than 0",
        "invalid_rows": int(
            order_items_clean["price"]
            .le(0)
            .sum()
        ),
    },
    {
        "table_name": "order_items",
        "check": "freight_value must not be negative",
        "invalid_rows": int(
            order_items_clean["freight_value"]
            .lt(0)
            .sum()
        ),
    },
    {
        "table_name": "payments",
        "check": "payment_sequential must be at least 1",
        "invalid_rows": int(
            payments_clean["payment_sequential"]
            .lt(1)
            .sum()
        ),
    },
    {
        "table_name": "payments",
        "check": (
            "payment_installments must not be negative"
        ),
        "invalid_rows": int(
            payments_clean["payment_installments"]
            .lt(0)
            .sum()
        ),
    },
    {
        "table_name": "payments",
        "check": "payment_value must not be negative",
        "invalid_rows": int(
            payments_clean["payment_value"]
            .lt(0)
            .sum()
        ),
    },
    {
        "table_name": "products",
        "check": (
            "Product description and photo fields "
            "must not be negative"
        ),
        "invalid_rows": int(
            products_clean[
                product_count_columns
            ]
            .lt(0)
            .any(axis=1)
            .sum()
        ),
    },
    {
        "table_name": "products",
        "check": (
            "Product physical attributes "
            "must not be negative"
        ),
        "invalid_rows": int(
            products_clean[
                product_physical_columns
            ]
            .lt(0)
            .any(axis=1)
            .sum()
        ),
    },
    {
        "table_name": "geolocation",
        "check": "Latitude must be between -90 and 90",
        "invalid_rows": int(
            (
                ~geolocation_clean[
                    "geolocation_lat"
                ].between(-90, 90)
            ).sum()
        ),
    },
    {
        "table_name": "geolocation",
        "check": (
            "Longitude must be between -180 and 180"
        ),
        "invalid_rows": int(
            (
                ~geolocation_clean[
                    "geolocation_lng"
                ].between(-180, 180)
            ).sum()
        ),
    },
]

numerical_validation = pd.DataFrame(
    numerical_checks
)

numerical_validation[
    "validation_passed"
] = numerical_validation["invalid_rows"].eq(0)

display(numerical_validation)

,table_name,check,invalid_rows,validation_passed
0,order_items,order_item_id must be at least 1,0,True
1,order_items,price must be greater than 0,0,True
2,order_items,freight_value must not be negative,0,True
3,payments,payment_sequential must be at least 1,0,True
4,payments,payment_installments must not be negative,0,True
5,payments,payment_value must not be negative,0,True
6,products,Product description and photo fields must not ...,0,True
7,products,Product physical attributes must not be negative,0,True
8,geolocation,Latitude must be between -90 and 90,0,True
9,geolocation,Longitude must be between -180 and 180,0,True


In [19]:
for _, row in numerical_validation.iterrows():
    passed = bool(row["validation_passed"])

    record_validation(
        area="Numerical domain",
        table_name=row["table_name"],
        check=row["check"],
        result="Passed" if passed else "Failed",
        details=(
            f"{int(row['invalid_rows']):,} invalid rows"
        ),
    )

## 14. Retained Temporal Anomalies

Temporal inconsistencies that could not be corrected reliably were retained during cleaning.

They are measured here and recorded as documented limitations rather than critical validation failures.

In [20]:
orders_clean = cleaned_datasets["orders"]
reviews_clean = cleaned_datasets["reviews"]

shipping_timeline = (
    cleaned_datasets["order_items"][
        [
            "order_id",
            "order_item_id",
            "shipping_limit_date",
        ]
    ]
    .merge(
        orders_clean[
            [
                "order_id",
                "order_purchase_timestamp",
            ]
        ],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
)

review_timeline = (
    reviews_clean[
        [
            "review_id",
            "order_id",
            "review_creation_date",
            "review_answer_timestamp",
        ]
    ]
    .merge(
        orders_clean[
            [
                "order_id",
                "order_purchase_timestamp",
            ]
        ],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
)

review_timeline["review_creation_day"] = (
    review_timeline["review_creation_date"]
    .dt.normalize()
)

review_timeline["purchase_day"] = (
    review_timeline["order_purchase_timestamp"]
    .dt.normalize()
)

temporal_checks = [
    {
        "table_name": "orders",
        "check": "Approval before purchase",
        "affected_rows": int(
            (
                orders_clean["order_approved_at"].notna()
                & (
                    orders_clean["order_approved_at"]
                    < orders_clean[
                        "order_purchase_timestamp"
                    ]
                )
            ).sum()
        ),
    },
    {
        "table_name": "orders",
        "check": "Carrier handoff before purchase",
        "affected_rows": int(
            (
                orders_clean[
                    "order_delivered_carrier_date"
                ].notna()
                & (
                    orders_clean[
                        "order_delivered_carrier_date"
                    ]
                    < orders_clean[
                        "order_purchase_timestamp"
                    ]
                )
            ).sum()
        ),
    },
    {
        "table_name": "orders",
        "check": "Customer delivery before purchase",
        "affected_rows": int(
            (
                orders_clean[
                    "order_delivered_customer_date"
                ].notna()
                & (
                    orders_clean[
                        "order_delivered_customer_date"
                    ]
                    < orders_clean[
                        "order_purchase_timestamp"
                    ]
                )
            ).sum()
        ),
    },
    {
        "table_name": "orders",
        "check": "Customer delivery before carrier handoff",
        "affected_rows": int(
            (
                orders_clean[
                    "order_delivered_customer_date"
                ].notna()
                & orders_clean[
                    "order_delivered_carrier_date"
                ].notna()
                & (
                    orders_clean[
                        "order_delivered_customer_date"
                    ]
                    < orders_clean[
                        "order_delivered_carrier_date"
                    ]
                )
            ).sum()
        ),
    },
    {
        "table_name": "orders",
        "check": "Estimated delivery before purchase",
        "affected_rows": int(
            (
                orders_clean[
                    "order_estimated_delivery_date"
                ]
                < orders_clean[
                    "order_purchase_timestamp"
                ]
            ).sum()
        ),
    },
    {
        "table_name": "order_items",
        "check": "Shipping limit before purchase",
        "affected_rows": int(
            (
                shipping_timeline[
                    "shipping_limit_date"
                ]
                < shipping_timeline[
                    "order_purchase_timestamp"
                ]
            ).sum()
        ),
    },
    {
        "table_name": "reviews",
        "check": "Survey sent before purchase date",
        "affected_rows": int(
            (
                review_timeline[
                    "review_creation_day"
                ].notna()
                & review_timeline[
                    "purchase_day"
                ].notna()
                & (
                    review_timeline[
                        "review_creation_day"
                    ]
                    < review_timeline[
                        "purchase_day"
                    ]
                )
            ).sum()
        ),
    },
    {
        "table_name": "reviews",
        "check": "Review answer before review creation",
        "affected_rows": int(
            (
                review_timeline[
                    "review_answer_timestamp"
                ].notna()
                & review_timeline[
                    "review_creation_date"
                ].notna()
                & (
                    review_timeline[
                        "review_answer_timestamp"
                    ]
                    < review_timeline[
                        "review_creation_date"
                    ]
                )
            ).sum()
        ),
    },
]

temporal_validation = pd.DataFrame(
    temporal_checks
)

display(temporal_validation)

,table_name,check,affected_rows
0,orders,Approval before purchase,0
1,orders,Carrier handoff before purchase,166
2,orders,Customer delivery before purchase,0
3,orders,Customer delivery before carrier handoff,23
4,orders,Estimated delivery before purchase,0
5,order_items,Shipping limit before purchase,0
6,reviews,Survey sent before purchase date,64
7,reviews,Review answer before review creation,0


In [21]:
shipping_range_validation = pd.DataFrame(
    [
        {
            "table_name": "order_items",
            "check": (
                "Shipping-limit date falls in 2019 "
                "or later"
            ),
            "affected_rows": int(
                (
                    cleaned_datasets["order_items"][
                        "shipping_limit_date"
                    ].notna()
                    & (
                        cleaned_datasets["order_items"][
                            "shipping_limit_date"
                        ]
                        >= pd.Timestamp("2019-01-01")
                    )
                ).sum()
            ),
        }
    ]
)

temporal_validation = pd.concat(
    [
        temporal_validation,
        shipping_range_validation,
    ],
    ignore_index=True,
)

display(temporal_validation)

,table_name,check,affected_rows
0,orders,Approval before purchase,0
1,orders,Carrier handoff before purchase,166
2,orders,Customer delivery before purchase,0
3,orders,Customer delivery before carrier handoff,23
4,orders,Estimated delivery before purchase,0
5,order_items,Shipping limit before purchase,0
6,reviews,Survey sent before purchase date,64
7,reviews,Review answer before review creation,0
8,order_items,Shipping-limit date falls in 2019 or later,4


In [22]:
for _, row in temporal_validation.iterrows():
    affected_rows = int(row["affected_rows"])

    record_validation(
        area="Temporal consistency",
        table_name=row["table_name"],
        check=row["check"],
        result=(
            "Passed"
            if affected_rows == 0
            else "Passed with limitation"
        ),
        details=(
            f"{affected_rows:,} affected rows retained"
        ),
        severity="Warning",
    )

## 15. Lookup and Geographic Coverage

Category translation and geolocation are incomplete lookup relationships.

They are validated for coverage but are not treated as strict foreign-key relationships.

In [23]:
products_clean = cleaned_datasets["products"]
translations_clean = cleaned_datasets[
    "category_translation"
]

product_categories = set(
    products_clean["product_category_name"]
    .dropna()
)

translated_categories = set(
    translations_clean["product_category_name"]
    .dropna()
)

untranslated_categories = sorted(
    product_categories - translated_categories
)

unused_translations = sorted(
    translated_categories - product_categories
)

products_in_untranslated_categories = int(
    products_clean["product_category_name"]
    .isin(untranslated_categories)
    .sum()
)

category_coverage = pd.DataFrame(
    [
        {
            "check": "Distinct product categories without translation",
            "affected_count": len(
                untranslated_categories
            ),
            "details": ", ".join(
                untranslated_categories
            ),
        },
        {
            "check": "Products in untranslated categories",
            "affected_count": products_in_untranslated_categories,
            "details": (
                "Portuguese category retained"
            ),
        },
        {
            "check": "Translations not used by products",
            "affected_count": len(
                unused_translations
            ),
            "details": ", ".join(
                unused_translations
            ),
        },
    ]
)

display(category_coverage)

,check,affected_count,details
0,Distinct product categories without translation,2,"pc_gamer, portateis_cozinha_e_preparadores_de_..."
1,Products in untranslated categories,13,Portuguese category retained
2,Translations not used by products,0,


In [24]:
record_validation(
    area="Lookup coverage",
    table_name="products",
    check="Product-category translation coverage",
    result=(
        "Passed"
        if not untranslated_categories
        else "Passed with limitation"
    ),
    details=(
        f"{len(untranslated_categories)} distinct "
        "categories and "
        f"{products_in_untranslated_categories:,} "
        "product rows lack an English translation"
    ),
    severity="Warning",
)

geolocation_zip_values = set(
    cleaned_datasets["geolocation"][
        "geolocation_zip_code_prefix"
    ]
)

zip_coverage_rows = []

for table_name, column_name in {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
}.items():
    source_values = cleaned_datasets[
        table_name
    ][column_name]

    unmatched_mask = ~source_values.isin(
        geolocation_zip_values
    )

    unmatched_rows = int(unmatched_mask.sum())

    unmatched_prefixes = int(
        source_values.loc[
            unmatched_mask
        ].nunique()
    )

    zip_coverage_rows.append(
        {
            "table_name": table_name,
            "column_name": column_name,
            "unmatched_rows": unmatched_rows,
            "unmatched_distinct_prefixes": (
                unmatched_prefixes
            ),
        }
    )

    record_validation(
        area="Geographic coverage",
        table_name=table_name,
        check="ZIP-prefix coverage in geolocation",
        result=(
            "Passed"
            if unmatched_rows == 0
            else "Passed with limitation"
        ),
        details=(
            f"{unmatched_rows:,} rows across "
            f"{unmatched_prefixes:,} ZIP prefixes "
            "have no geolocation match"
        ),
        severity="Warning",
    )

zip_coverage = pd.DataFrame(
    zip_coverage_rows
)

display(zip_coverage)

,table_name,column_name,unmatched_rows,unmatched_distinct_prefixes
0,customers,customer_zip_code_prefix,278,157
1,sellers,seller_zip_code_prefix,7,7


## 16. Join-Grain Reference

The validated table grains must be preserved during database design and analysis.

Multiple one-to-many tables must not be joined directly to orders without first controlling their grain.

In [25]:
join_grain_reference = pd.DataFrame(
    [
        {
            "table_name": "customers",
            "validated_grain": (
                "One row per customer_id"
            ),
            "validated_key": "customer_id",
            "join_guidance": (
                "Use customer_unique_id only for "
                "cross-order customer analysis"
            ),
        },
        {
            "table_name": "orders",
            "validated_grain": "One row per order",
            "validated_key": "order_id",
            "join_guidance": (
                "Base table for order-level analysis"
            ),
        },
        {
            "table_name": "order_items",
            "validated_grain": (
                "One product item within an order"
            ),
            "validated_key": (
                "order_id + order_item_id"
            ),
            "join_guidance": (
                "Aggregate to order level before joining "
                "to other one-to-many order tables"
            ),
        },
        {
            "table_name": "payments",
            "validated_grain": (
                "One payment record within an order"
            ),
            "validated_key": (
                "order_id + payment_sequential"
            ),
            "join_guidance": (
                "Aggregate to order level before joining "
                "to order items or reviews"
            ),
        },
        {
            "table_name": "reviews",
            "validated_grain": (
                "One review record associated with an order"
            ),
            "validated_key": "review_id + order_id",
            "join_guidance": (
                "Define a review-level or order-level rule "
                "before joining to items or payments"
            ),
        },
        {
            "table_name": "products",
            "validated_grain": "One row per product",
            "validated_key": "product_id",
            "join_guidance": (
                "Safe many-to-one join from order items"
            ),
        },
        {
            "table_name": "sellers",
            "validated_grain": "One row per seller",
            "validated_key": "seller_id",
            "join_guidance": (
                "Safe many-to-one join from order items"
            ),
        },
        {
            "table_name": "geolocation",
            "validated_grain": (
                "One geographic coordinate observation"
            ),
            "validated_key": "No source primary key",
            "join_guidance": (
                "Create one representative row per ZIP "
                "before customer or seller joins"
            ),
        },
        {
            "table_name": "category_translation",
            "validated_grain": (
                "One row per translated Portuguese category"
            ),
            "validated_key": "product_category_name",
            "join_guidance": (
                "Use a left join because lookup coverage "
                "is incomplete"
            ),
        },
    ]
)

display(join_grain_reference)

,table_name,validated_grain,validated_key,join_guidance
0,customers,One row per customer_id,customer_id,Use customer_unique_id only for cross-order cu...
1,orders,One row per order,order_id,Base table for order-level analysis
2,order_items,One product item within an order,order_id + order_item_id,Aggregate to order level before joining to oth...
3,payments,One payment record within an order,order_id + payment_sequential,Aggregate to order level before joining to ord...
4,reviews,One review record associated with an order,review_id + order_id,Define a review-level or order-level rule befo...
5,products,One row per product,product_id,Safe many-to-one join from order items
6,sellers,One row per seller,seller_id,Safe many-to-one join from order items
7,geolocation,One geographic coordinate observation,No source primary key,Create one representative row per ZIP before c...
8,category_translation,One row per translated Portuguese category,product_category_name,Use a left join because lookup coverage is inc...


## 17. Final Validation Summary

The final readiness decision is based on critical validation checks.

Warnings and documented limitations remain visible but do not prevent database design unless they conflict with a proposed constraint or analytical use.

In [26]:
validation_results_df = pd.DataFrame(
    validation_results
)

result_order = {
    "Failed": 0,
    "Passed with limitation": 1,
    "Passed": 2,
}

severity_order = {
    "Critical": 0,
    "Warning": 1,
}

validation_results_df["result_order"] = (
    validation_results_df["result"]
    .map(result_order)
)

validation_results_df["severity_order"] = (
    validation_results_df["severity"]
    .map(severity_order)
)

validation_results_df = (
    validation_results_df
    .sort_values(
        [
            "severity_order",
            "result_order",
            "validation_area",
            "table_name",
        ]
    )
    .drop(
        columns=[
            "result_order",
            "severity_order",
        ]
    )
    .reset_index(drop=True)
)

display(validation_results_df)

,validation_area,table_name,check,severity,result,details
0,Categorical domain,customers,customer_state uses valid Brazilian state codes,Critical,Passed,No invalid state codes
1,Categorical domain,geolocation,geolocation_state uses valid Brazilian state c...,Critical,Passed,No invalid state codes
2,Categorical domain,orders,order_status allowed values,Critical,Passed,No unexpected values
3,Categorical domain,payments,payment_type allowed values,Critical,Passed,No unexpected values
4,Categorical domain,reviews,Review scores between 1 and 5,Critical,Passed,0 invalid rows
5,Categorical domain,sellers,seller_state uses valid Brazilian state codes,Critical,Passed,No invalid state codes
6,Data types,category_translation,Expected cleaned data types,Critical,Passed,All expected types matched
7,Data types,customers,Expected cleaned data types,Critical,Passed,All expected types matched
8,Data types,geolocation,Expected cleaned data types,Critical,Passed,All expected types matched
9,Data types,order_items,Expected cleaned data types,Critical,Passed,All expected types matched


In [27]:
validation_status_summary = (
    validation_results_df
    .groupby(
        ["severity", "result"],
        dropna=False,
    )
    .size()
    .rename("check_count")
    .reset_index()
)

display(validation_status_summary)

,severity,result,check_count
0,Critical,Passed,126
1,Warning,Passed,8
2,Warning,Passed with limitation,12


In [28]:
critical_failures = validation_results_df.loc[
    (
        validation_results_df["severity"]
        == "Critical"
    )
    & (
        validation_results_df["result"]
        == "Failed"
    )
]

documented_limitations = validation_results_df.loc[
    validation_results_df["result"]
    == "Passed with limitation"
]

database_ready = critical_failures.empty

readiness_summary = pd.DataFrame(
    [
        {
            "critical_checks": int(
                (
                    validation_results_df["severity"]
                    == "Critical"
                ).sum()
            ),
            "critical_failures": len(
                critical_failures
            ),
            "documented_limitations": len(
                documented_limitations
            ),
            "database_readiness": (
                "Ready with documented limitations"
                if database_ready
                else "Not ready"
            ),
        }
    ]
)

display(readiness_summary)

,critical_checks,critical_failures,documented_limitations,database_readiness
0,126,0,12,Ready with documented limitations


In [29]:
if not critical_failures.empty:
    display(critical_failures)

    raise AssertionError(
        "Critical validation failures were detected. "
        "Resolve them before database design."
    )

print(
    "Cleaned-data validation passed. "
    "The datasets are ready for database design "
    "with the documented limitations."
)

Cleaned-data validation passed. The datasets are ready for database design with the documented limitations.


## Final Validation Conclusion

The cleaned Olist datasets passed all critical validation checks.

The validation confirmed:

- Expected row and column counts
- Correct cleaned schemas and data types
- Complete and unique primary and composite keys
- No exact duplicate records
- Valid core parent-child relationships
- Preserved missing-value behaviour
- Successful structural and formatting transformations
- Valid categorical and numerical domains

Several source-data limitations remain documented, including incomplete order-related records, temporal anomalies, untranslated product categories, and incomplete geolocation coverage. These records were retained because no reliable correction could be made without introducing unsupported assumptions.

The cleaned datasets are therefore considered **ready for database design and loading with documented limitations**.

The next stage is `04_database_design`.